In [0]:
%sql
SELECT pl.*, mp.person_id FROM 5_projects.pharos_20260522.pharos_pilot_list AS pl
LEFT JOIN 4_prod.raw.mill_person AS mp
ON pl.research_id = mp.person_ID


In [0]:
%sql
SELECT * FROM 4_prod.pacs_dlt.pacs_examcode_dict
WHERE (
    preferred ILIKE 'MRI breast%' AND preferred != 'MRI Breast implant Both'
) OR (
    preferred ILIKE 'XR Mammogram%'
) OR (
    preferred ILIKE 'US Breast%'
) OR (
    preferred ILIKE 'US Axilla%'
)


In [0]:
%sql
CREATE TABLE 5_projects.pharos_20260522.extract_imaging AS
WITH pid AS (
    SELECT DISTINCT research_id AS person_id FROM 5_projects.pharos_20260522.pharos_pilot_list 
),
ec AS (
    SELECT DISTINCT short_code FROM 4_prod.pacs_dlt.pacs_examcode_dict
    WHERE (
        preferred ILIKE 'MRI breast%' AND preferred != 'MRI Breast implant Both'
    ) OR (
        preferred ILIKE 'XR Mammogram%'
    ) OR (
        preferred ILIKE 'US Breast%'
    ) OR (
        preferred ILIKE 'US Axilla%'
    )
)
SELECT * FROM 4_prod.pacs.imaging_metadata AS im
INNER JOIN pid on pid.person_id = im.personid
INNER JOIN ec on ec.short_code = im.examcode


In [0]:
%sql

SELECT DISTINCT AccessionNbr FROM 5_projects.pharos_20260522.extract_imaging

In [0]:
%sql
CREATE OR REPLACE TABLE 5_projects.pharos_20260522.imaging_report AS
SELECT DISTINCT ir.PersonId, ir.ReportEventId, ir.AccessionNbr, bd.AnonymizedText FROM 4_prod.pacs.imaging_report AS ir
LEFT JOIN 4_prod.rde.rde_blobdataset AS bd
ON ir.reporteventid = bd.eventid
INNER JOIN (SELECT DISTINCT AccessionNbr FROM 5_projects.pharos_20260522.extract_imaging) AS an
ON an.AccessionNbr = ir.AccessionNbr

In [0]:
%sql
SELECT COUNT(DISTINCT PersonId), COUNT(ReportEventId), COUNT(DISTINCT ReportEventId) FROM 5_projects.pharos_20260522.imaging_report
LIMIT 300

In [0]:
dbutils.fs.cp(
       "dbfs:/Volumes/1_inland/sectra/vone/pharos_20260522/",
       "dbfs:/Volumes/7_outgoing/pharos/imaging/pharos_20260522/",
       recurse=True
   )